In [ ]:
import os
from tqdm import tqdm
import polars as pl
from statsmodels.stats import multitest
from plotnine import *

## Genes2Keep

In [ ]:
# Burden test genes

res_dir = 'PATH_TO_FILE'
n_phenos = 127
burdens = "lofteeHC_mac20"
filename = f'regenie_{n_phenos}phenotypes_{burdens}.parquet'

!dx download project-REDACTED:/processed_data/association_files/{filename} -o {res_dir}/{filename}

r = pl.read_parquet(f'{res_dir}/regenie_{n_phenos}phenotypes_{burdens}.parquet')
bt_genes = r.filter(pl.col('pval_fdr') < 0.05).sort('pval')['region'].unique().to_list()
len(bt_genes)

In [ ]:
# Burden test genes
burdens = "lofteeHC_maf1e-3"
filename = f'regenie_{n_phenos}phenotypes_{burdens}.parquet'

!dx download project-REDACTED:/processed_data/association_files/{filename} -o {res_dir}/{filename}

r = pl.read_parquet(f'{res_dir}/regenie_{n_phenos}phenotypes_{burdens}.parquet')
bt_genes_maf1e3 = r.filter(pl.col('pval_fdr') < 0.05).sort('pval')['region'].unique().to_list()

bt_genes = list(set(bt_genes_maf1e3).union(set(bt_genes)))
len(bt_genes)

In [ ]:
# Olink genes

!dx download project-REDACTED:/processed_data/olink/preprocessed/proteomics_genes.txt

olink_genes = pl.read_csv('proteomics_genes.txt', has_header=False).select(pl.col('column_1').unique()).to_series().to_list()
len(olink_genes)

## Read Gencode file

In [ ]:
!dx download project-REDACTED:/processed_data/misc_data/gencode.v40.annotation.gtf.gz -o PATH_TO_FILE

In [ ]:
gtf_path = "PATH_TO_FILE"

gtf_pl = pl.read_csv(
    gtf_path,
    separator="\t",
    comment_prefix="#",
    has_header=False,
    new_columns=["Chromosome", "source", "Feature", "Start", "End", "score", "Strand", "frame", "attributes"]
)

gtf_pl = gtf_pl.with_row_index("row_nr")
gtf_pl

In [ ]:
atts = (
    gtf_pl.select("row_nr", "attributes")
    
    .with_columns(
        attrs_list=pl.col("attributes").str.split("; ")
    )
    .explode("attrs_list")
    
    .with_columns(
        pl.col("attrs_list")
        .str.split_exact(" ", 1)
        .struct.rename_fields(["attribute", "value"])
        .alias("fields")
    ).unnest("fields")

    .with_columns(
        pl.col("value").str.strip_chars('"')
    )

    .pivot(
        index="row_nr",
        on="attribute",
        values="value",
        aggregate_function=pl.element().implode()
    )

    .with_columns(
        pl.col(pl.List).list.join(", ").str.strip_chars(";").replace("", None)
    )

    .with_columns(
        level = pl.col('level').cast(pl.Int32),
        exon_number = pl.col('exon_number').cast(pl.Int32),
    )
)

atts

In [ ]:
window_size = 10_000
window_size = 5_000
gtf_genes = (
    gtf_pl
    .drop('attributes')
    .join(atts, on='row_nr')
    .with_columns(
        region = pl.col('gene_id').str.split('.').list.get(0),
        gene_start_pad = pl.col('Start') - window_size,
        gene_end_pad = pl.col('End') + window_size,
    )

    .filter(
        (pl.col('Feature') == 'gene') &
        (pl.col('region').is_in(bt_genes + olink_genes))
    )

    .select(['region', 'Chromosome', 'Start', 'End', 'gene_start_pad', 'gene_end_pad', 'Strand', 'gene_name', 'gene_type'])

    .rename({
        'Chromosome': 'chr',
        'Start': 'gene_start',
        'End': 'gene_end',
        'Strand': 'gene_strand',
    })
)

gtf_genes

In [ ]:
gtf_genes['region'].n_unique()

In [ ]:
# 131 genes are not in the gencode file
missing_genes = set(olink_genes).union(set(bt_genes)) - set(gtf_genes['region'].unique())
len(missing_genes)

## Intersect with region files from the RAP

In [ ]:
!dx download project-REDACTED:/processed_data/wgs/all_region_files.parquet -o PATH_TO_FILE

In [ ]:
reg_files = (
    pl.read_parquet('PATH_TO_FILE')
    .drop('__index_level_0__')
    .with_columns(
        file_num = pl.col('file_name').str.split('.').list.get(0).str.slice(17)
    )
)
reg_files

In [ ]:
# 1. Perform an inner join on 'chr'
# This matches genes to ALL files on the same chromosome first.
# We add a suffix to avoid column name collisions if both DFs have 'start'/'end' columns.
matches = (
    gtf_genes.join(reg_files, on="chr", suffix="_file")
    .filter(
        # 2. Filter for Interval Overlap
        # A gene overlaps a file if:
        # (Gene Start <= File End) AND (Gene End >= File Start)
        (pl.col("gene_start_pad") <= pl.col("end")) & 
        (pl.col("gene_end_pad") >= pl.col("start"))
    )
)

# 3. Extract the unique list of files to process
files_to_process = matches.select(["file_name", 'file_num']).unique()

# Optional: View the result
print(f"Found {files_to_process.height} files covering the target genes.")
# files_to_process.write_csv('PATH_TO_FILE', separator='\t')
files_to_process

In [ ]:
matches

In [ ]:
# total number of genes that couldn't be matched
len(set(bt_genes).union(set(olink_genes)) - set(matches['region'].unique()))

In [ ]:
len(set(bt_genes) - set(matches['region'].unique()))

In [ ]:
len(set(olink_genes) - set(matches['region'].unique())) # those genes that cannot be matched are all olink gens

In [ ]:
len(set(bt_genes).union(set(olink_genes)) & set(matches['region'].unique()))

In [ ]:
files2download = sorted([f"project-REDACTED:/processed_data/wgs/qced_maf1e-3.parquet/norm_qced_ukb24310_{fnum}.parquet" for fnum in files_to_process['file_num'].unique()])
files2download

In [ ]:
matched_genes = matches.select('region').unique()
print(len(matched_genes))

In [ ]:
matched_genes.write_parquet('matched_regions.parquet')
! dx upload matched_regions.parquet --path  project-REDACTED:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/

In [ ]:
import subprocess
from concurrent.futures import ThreadPoolExecutor
from tqdm.notebook import tqdm  # Progress bar

# --- CONFIGURATION ---
OUTPUT_DIR = "PATH_TO_FILE"      # Folder to save files to
MAX_WORKERS = 8                 # Number of parallel downloads (Don't go too high or you'll hit API limits)
# ---------------------

# Ensure output directory exists
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

def download_file(file_identifier):
    """
    Runs the dx download command for a single file.
    """
    try:
        # Construct the command
        # -f: Force overwrite (optional)
        # -o: Output directory
        cmd = ["dx", "download", file_identifier, "-o", OUTPUT_DIR, "-f"]
        
        # Run command and capture output
        result = subprocess.run(
            cmd, 
            capture_output=True, 
            text=True, 
            check=True
        )
        return True, file_identifier
    except subprocess.CalledProcessError as e:
        # Return False and the error message if it fails
        return False, f"{file_identifier}: {e.stderr}"

# Run the downloads in parallel
print(f"Starting download of {len(files2download)} files with {MAX_WORKERS} threads...")

failed_files = []

# ThreadPoolExecutor manages the pool of worker threads
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    # Map the function to the file list and wrap in tqdm for a progress bar
    results = list(tqdm(executor.map(download_file, files2download), total=len(files2download)))

# Process results
successful = sum(1 for status, _ in results if status)
for status, msg in results:
    if not status:
        failed_files.append(msg)

print(f"\nDownload Complete.")
print(f"Success: {successful}")
print(f"Failed:  {len(failed_files)}")

if failed_files:
    print("\nErrors:", failed_files[:5]) # Show first 5 errors

In [ ]:
!dx download project-REDACTED:/processed_data/wgs/qced_maf1e-3.parquet/norm_qced_ukb24310_c1_b8634_v1.parquet -o {OUTPUT_DIR}

In [ ]:
!dx download project-REDACTED:/processed_data/wgs/qced_maf1e-3.parquet/norm_qced_ukb24310_c1_b8674_v1.parquet -o {OUTPUT_DIR}

In [ ]:
!dx download project-REDACTED:/processed_data/wgs/qced_maf1e-3.parquet/norm_qced_ukb24310_c2_b1848_v1.parquet -o {OUTPUT_DIR}